# Phase 1: decision tree & random forest vs sklearn

From-scratch CART decision tree and random forest validated against sklearn on the BACE dataset (Morgan fingerprints). Classification targets the binary active/inactive label; regression targets continuous pIC50, the same tree and forest code, with only the task (impurity + leaf value + aggregation) swapped.

Each model is fit alongside its sklearn equivalent on one shared split per objective, so any difference is the algorithm, not the data.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()                      # put the project root (contains core/) on the path
while not (root / "core").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

from core.data import get_dataset
from core.RF.tree import DecisionTree, REGRESSION
from core.RF.forest import RandomForest

## Data

Load the cached BACE features once. Classification uses the binary Class label on a stratified split; regression uses continuous pIC50. Each objective's models share one split.

In [2]:
X, y_class, y_reg = get_dataset()
y_c = y_class.astype(int)

print("X:", X.shape)
print("active / inactive:", np.bincount(y_c))
print("pIC50 range:", round(float(y_reg.min()), 2), "to", round(float(y_reg.max()), 2))

X: (1513, 2048)
active / inactive: [822 691]
pIC50 range: 2.54 to 10.52


## Classification — active / inactive

Binary label, gini impurity + majority-class leaves. The forest votes across 100 trees.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_c, test_size=0.2, random_state=0, stratify=y_c,
)

In [4]:
ours = DecisionTree().fit(X_train, y_train)
skl = DecisionTreeClassifier(criterion="gini", random_state=0).fit(X_train, y_train)
ours_pred, skl_pred = ours.predict(X_test), skl.predict(X_test)

ctree_ours = accuracy_score(y_test, ours_pred)
ctree_skl = accuracy_score(y_test, skl_pred)
ctree_agree = (ours_pred == skl_pred).mean()
print(f"ours {ctree_ours:.4f}   sklearn {ctree_skl:.4f}   agreement {ctree_agree:.4f}")

ours 0.7921   sklearn 0.8020   agreement 0.9439


In [5]:
ours = RandomForest(n_estimators=100, random_state=0).fit(X_train, y_train)
skl = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
ours_pred, skl_pred = ours.predict(X_test), skl.predict(X_test)

cforest_ours = accuracy_score(y_test, ours_pred)
cforest_skl = accuracy_score(y_test, skl_pred)
cforest_agree = (ours_pred == skl_pred).mean()
print(f"ours {cforest_ours:.4f}   sklearn {cforest_skl:.4f}   agreement {cforest_agree:.4f}")

ours 0.8317   sklearn 0.8284   agreement 0.9571


In [6]:
pd.DataFrame(
    {"ours": [ctree_ours, cforest_ours],
     "sklearn": [ctree_skl, cforest_skl],
     "agreement": [ctree_agree, cforest_agree]},
    index=["decision tree", "random forest"],
).round(4)

,ours,sklearn,agreement
decision tree,0.7921,0.8020,0.9439
random forest,0.8317,0.8284,0.9571


## Regression — pIC50

Continuous target, variance impurity + mean leaves, the same code with task=REGRESSION. Scored with R-squared and RMSE.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=0,
)

In [8]:
ours = DecisionTree(task=REGRESSION).fit(X_train, y_train)
skl = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
ours_pred, skl_pred = ours.predict(X_test), skl.predict(X_test)

rtree_ours_r2, rtree_ours_rmse = r2_score(y_test, ours_pred), np.sqrt(mean_squared_error(y_test, ours_pred))
rtree_skl_r2, rtree_skl_rmse = r2_score(y_test, skl_pred), np.sqrt(mean_squared_error(y_test, skl_pred))
print(f"ours    R2 {rtree_ours_r2:.4f}   RMSE {rtree_ours_rmse:.4f}")
print(f"sklearn R2 {rtree_skl_r2:.4f}   RMSE {rtree_skl_rmse:.4f}")

ours    R2 0.4743   RMSE 0.9676
sklearn R2 0.4342   RMSE 1.0039


In [9]:
ours = RandomForest(n_estimators=100, random_state=0, task=REGRESSION).fit(X_train, y_train)
skl = RandomForestRegressor(n_estimators=100, random_state=0).fit(X_train, y_train)
ours_pred, skl_pred = ours.predict(X_test), skl.predict(X_test)

rforest_ours_r2, rforest_ours_rmse = r2_score(y_test, ours_pred), np.sqrt(mean_squared_error(y_test, ours_pred))
rforest_skl_r2, rforest_skl_rmse = r2_score(y_test, skl_pred), np.sqrt(mean_squared_error(y_test, skl_pred))
print(f"ours    R2 {rforest_ours_r2:.4f}   RMSE {rforest_ours_rmse:.4f}")
print(f"sklearn R2 {rforest_skl_r2:.4f}   RMSE {rforest_skl_rmse:.4f}")

ours    R2 0.7034   RMSE 0.7268
sklearn R2 0.7049   RMSE 0.7249


In [10]:
pd.DataFrame(
    {"ours R2": [rtree_ours_r2, rforest_ours_r2],
     "sklearn R2": [rtree_skl_r2, rforest_skl_r2],
     "ours RMSE": [rtree_ours_rmse, rforest_ours_rmse],
     "sklearn RMSE": [rtree_skl_rmse, rforest_skl_rmse]},
    index=["decision tree", "random forest"],
).round(4)

,ours R2,sklearn R2,ours RMSE,sklearn RMSE
decision tree,0.4743,0.4342,0.9676,1.0039
random forest,0.7034,0.7049,0.7268,0.7249


## Analysis
**Validation**
Both models sit close to sklearn on the sample data and splits. Classification was within ~1 pt of accuracy; regression was nearly matching on random forest. Closeness is what validates the model implementation. An error would open a clear prediction accuracy gap which would lead to numbers dissimilar to a production library. 

**Why they're not identical**
Greedy splits and equal gain impurity minimization ties lead to differences in accuracy. Sklearn breaks equal gain ties in its own way compared to me. Prediction accuracy can range comparing tree to tree or forest to forest due to the tie-breaking rules, neither model is better due to this difference.

**Variance reduction (forest vs single tree)**
Classification accuracy: ~80% for single tree -> ~83% for forest. Agreement between models grew ~1% as well.
Regression accuracy: R-squared ~0.45, RMSE ~0.98 for single tree -> R-squared ~.70, RMSE ~.72 for forest.

Why? Bagging and per-split feature subsampling decorrelate the trees and make it so a collective ensemble of trees predicts better than any single one (Variance is decreased). Bagging: selecting a random bootstrap sample of molecules to train a tree on. Feature subsampling: selecting a random subsample of features (fingerprint bits) to test on per split generating a tree.

**Ensembling converges the two implementations**
Single trees diverge more (classification agreement ≈ ~94%; regression tree R² gap ≈ ~0.04)
Forests converge tighter (agreement ≈ ~96%; regression forest R² gap ≈ ~0.0015)
Why? Voting (classification) and averaging (regression) wash out per-tree tie-breaking, it shows on both classification and regression. Not just a fluke of one run.